# Full dataset audit and preparation

## Purpose and execution order

Audit every original record and produce consistent modelling partitions without inventing outcomes. The shared cleaning code processes 100,000-row chunks. A disk-backed duplicate index preserves state across chunks, so identical records are checked across the whole source.

After root setup, the next code cell checks whether an existing audit matches the current source hash and whether prepared splits exist. It reuses compatible preparation outputs or runs preparation when needed. The following cell displays the before/after missingness table.

**Before rerunning:** the full raw source must be available. Cached notebook outputs can be read without rerunning the cells. The browser-upload package does not include full processed partitions.

Code-cell outputs are retained from the completed real-data run; this documentation update does not rerun or alter their computation.


In [1]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/schema.py').exists())
sys.path.insert(0, str(ROOT))
def run_stage(name):
    completed = subprocess.run([sys.executable, str(ROOT / 'scripts' / name)], cwd=ROOT, capture_output=True, text=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
print('Project directory:', ROOT.name)


Project directory: flight_disruption_project


## Decision sequence

1. Normalize missing tokens and categorical text, then convert numeric types.
2. Validate clock formats, numeric domains and calendar dates, recording every counter.
3. Preserve valid early-arrival values and severe positive delays; IQR diagnostics are review aids, not an automatic removal rule.
4. Distinguish IATA airport codes from unresolved BTS numeric identifiers.
5. Derive the disruption target from cancellation, diversion and known arrival delay. Quarantine unknown or contradictory outcomes instead of filling a target.
6. Reconcile all rows, then assign chronological partitions and embargo dates.

Predictor imputation is not performed using full-data statistics at this stage. Numerical medians and encoders are fitted inside model-training folds later.


In [2]:
from src.common import sha256_file
audit_path = ROOT / 'reports/quality_audit.json'
reuse = audit_path.exists() and json.loads(audit_path.read_text()).get('source_sha256') == sha256_file(ROOT / 'data/raw/flights.csv')
if not reuse:
    run_stage('prepare_data.py')
else:
    print('Verified source checksum; reading the completed full-data audit.')
audit = json.loads((ROOT / 'reports/quality_audit.json').read_text())
print(audit['status'])
print(audit['split_counts'])
print({k:v for k,v in audit['counts'].items() if v and not k.startswith('raw_missing__')})

Verified source checksum; reading the completed full-data audit.
REAL_DATA_PREPARED
{'train': 3888115, 'validation': 923321, 'test': 947202, 'embargo': 60441}
{'raw_rows': 5819079, 'midnight_2400__SCHEDULED_ARRIVAL': 2, 'midnight_2400__DEPARTURE_TIME': 513, 'midnight_2400__ARRIVAL_TIME': 2456, 'midnight_2400__WHEELS_ON': 2005, 'midnight_2400__WHEELS_OFF': 727, 'negative_delay_retained__ARRIVAL_DELAY': 3500899, 'extreme_delay_over_1440_retained__ARRIVAL_DELAY': 32, 'negative_delay_retained__DEPARTURE_DELAY': 3277948, 'extreme_delay_over_1440_retained__DEPARTURE_DELAY': 32, 'numeric_airport_ids__ORIGIN_AIRPORT': 486165, 'unresolved_airport_ids__ORIGIN_AIRPORT': 486165, 'numeric_airport_ids__DESTINATION_AIRPORT': 486165, 'unresolved_airport_ids__DESTINATION_AIRPORT': 486165, 'eligible_rows': 5819079}


In [3]:
display(pd.read_csv(ROOT / 'reports/missingness_before_after.csv'))

,column,raw_rows,raw_missing,raw_missing_pct,eligible_rows,clean_missing,clean_missing_pct
0,YEAR,5819079,0,0.000000,5819079,0,0.000000
1,MONTH,5819079,0,0.000000,5819079,0,0.000000
2,DAY,5819079,0,0.000000,5819079,0,0.000000
3,DAY_OF_WEEK,5819079,0,0.000000,5819079,0,0.000000
4,AIRLINE,5819079,0,0.000000,5819079,0,0.000000
5,FLIGHT_NUMBER,5819079,0,0.000000,5819079,0,0.000000
6,TAIL_NUMBER,5819079,14721,0.252978,5819079,14721,0.252978
7,ORIGIN_AIRPORT,5819079,0,0.000000,5819079,0,0.000000
8,DESTINATION_AIRPORT,5819079,0,0.000000,5819079,0,0.000000
9,SCHEDULED_DEPARTURE,5819079,0,0.000000,5819079,0,0.000000


## Recorded findings and interpretation

All 5,819,079 records were retained; exact duplicates, conflicting business keys and quarantined rows were all zero. Missing arrival delay affected 105,071 rows, matching cancellations plus diversions in this source. The six missing training scheduled-duration values were later imputed with the fitted median of 122 minutes.

October contains 486,165 numeric origin IDs and the same number of numeric destination IDs. The `BTS_` namespace preserves their format without claiming that airport identities are reconciled. Negative delays are valid early arrivals: 3,500,899 negative arrival delays were retained, as were 32 arrival delays above 1,440 minutes.

Check that eligible rows equal training + validation + test + embargo rows. Read the audit and imputation tables together: retained structural absence in the audit is compatible with a complete numerical matrix after training-fitted preprocessing. Continue to **03_eda_and_features.ipynb**.
